# Week 4 Day 5 - the Sidekick

This is the project the whole week has been building towards. The Sidekick is a personal co-worker you can give a task and a definition of success, and it will work away using a real browser, a filesystem, web search and more, until it has met your criteria or needs to ask you something.

It is also where the stack-not-a-ladder idea pays off. The Sidekick's worker is a single create_agent from Layer 3, but we wrap it in our own loop rather than letting a framework run the show, and we reach down to Layer 1 tools and out to MCP servers as we please. You pick the altitude that suits each part of the job.

## How it is built

The worker is one `create_agent` with all the tools. Around it we run a small loop of our own:

1. The worker attempts the task.
2. An evaluator, which is just a model with structured output, checks the answer against your success criteria.
3. If the criteria are met, or the worker has a question, we stop and show you the result. Otherwise we hand the feedback back to the worker and let it try again.

We will build up to it in three steps. First the simplest possible version with no evaluator, then human-in-the-loop with middleware, then the full Sidekick from the project modules.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Before you run this</h2>
            <span style="color:#ff7800;">This lab uses <code>OPENAI_API_KEY</code>, <code>SERPER_API_KEY</code>, and your Pushover keys, and the full Sidekick brings up the headful browser through Node and npx. The browser will take over your screen for a few seconds when it runs. The project files <code>sidekick.py</code>, <code>sidekick_tools.py</code> and <code>app.py</code> sit next to this notebook.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# Imports and environment first

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from sidekick_tools import search, send_push_notification, wikipedia_lookup
from sidekick import Sidekick

load_dotenv(override=True)

## Step 1: the simplest Sidekick, with no evaluator

At its plainest, the Sidekick is a create_agent with a few tools and a memory, plus a tiny function that runs one turn of conversation. There is no evaluator here, so the worker simply does its best and replies. For many tasks this simple version is all you need, and the evaluator loop in Step 3 adds a check-and-retry layer on top of it.

In [ ]:
simple_worker = create_agent(
    model=ChatOpenAI(model="gpt-5.4-mini"),
    tools=[search, send_push_notification, wikipedia_lookup],
    system_prompt="You are Sidekick, a helpful personal assistant. Use your tools to complete the task.",
    checkpointer=InMemorySaver(),
)

async def ask(worker, message):
    config = {"configurable": {"thread_id": "simple-sidekick"}}
    result = await worker.ainvoke({"messages": [{"role": "user", "content": message}]}, config=config)
    return result["messages"][-1].content

reply = await ask(simple_worker, "Search for who won the Nobel Prize in Physics in 2023 and summarise it in two lines.")
print(reply)

## Step 2: human-in-the-loop with middleware

Some actions deserve a human nod before they happen. In LangChain 1.0 this is a piece of middleware. We tell `HumanInTheLoopMiddleware` which tools to pause on, and the agent stops and waits for our decision instead of running them.

When the agent reaches such a tool, the run pauses and returns an interrupt that describes the pending action. We can then approve, edit, reject, or respond, and resume.

In [ ]:
from langchain_core.tools import tool

@tool
def book_meeting(person: str, day: str) -> str:
    """Book a meeting with a person on a given day."""
    return f"Meeting booked with {person} on {day}."

approval_agent = create_agent(
    model=ChatOpenAI(model="gpt-5.4-mini"),
    tools=[book_meeting],
    system_prompt="You are a scheduling assistant. Use the book_meeting tool.",
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"book_meeting": True})],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "approval-demo"}}
result = await approval_agent.ainvoke(
    {"messages": [{"role": "user", "content": "Book a meeting with Sam on Friday."}]}, config
)

interrupt = result["__interrupt__"][0]
print("The agent paused and is asking for approval:")
print(interrupt.value["action_requests"][0]["description"])

The agent is waiting. We approve the action, and the run continues from exactly where it paused.

In [ ]:
resumed = await approval_agent.ainvoke(Command(resume={"decisions": [{"type": "approve"}]}), config)
print(resumed["messages"][-1].content)

## Step 3: the full Sidekick, with the evaluator loop

The complete version lives in `sidekick.py` and `sidekick_tools.py`. The worker has the full toolkit: a headful browser and a sandbox filesystem through MCP servers, plus web search, Wikipedia and push notifications. The evaluator loop checks each answer against your success criteria and pushes the worker to try again when needed.

This is the whole stack in one object. The worker is a Layer 3 create_agent, its tools are Layer 1 `@tool` functions alongside MCP servers, its memory is a Layer 2 checkpointer, and the loop around it is code you wrote yourself. You chose the altitude for each part.

Let us bring one to life and give it a task that uses the browser. Watch the window open and drive itself. The top story on Hacker News changes through the day, so you will see a different headline than the one shown here.

In [ ]:
sidekick = Sidekick()
await sidekick.setup()
print(f"Sidekick ready with {len(sidekick.tools)} tools")

In [ ]:
history = await sidekick.run_superstep(
    message="Go to Hacker News at news.ycombinator.com and tell me the title of the current top story.",
    success_criteria="The reply names a specific story currently on the Hacker News front page.",
    history=[],
)
for entry in history:
    print(f"[{entry['role']}] {entry['content'][:200]}\n")

## The Gradio app

`app.py` wraps all of this in a chat interface with two boxes, one for your request and one for your success criteria. You can run it from a terminal with `uv run app.py`. The cell below builds a trimmed version of the same interface, with a single Go button, so you can try it straight from the notebook. The full `app.py` adds a Reset button and tidies up the browser when you close the page.

In [ ]:
import gradio as gr

async def setup_ui():
    new_sidekick = Sidekick()
    await new_sidekick.setup()
    return new_sidekick

async def process(sk, message, success_criteria, history):
    results = await sk.run_superstep(message, success_criteria, history)
    return results, sk

with gr.Blocks(title="Sidekick") as ui:
    gr.Markdown("## Sidekick Personal Co-Worker")
    sk_state = gr.State()
    with gr.Row():
        chatbot = gr.Chatbot(label="Sidekick", height=300)
    with gr.Group():
        with gr.Row():
            message = gr.Textbox(show_label=False, placeholder="Your request to the Sidekick")
        with gr.Row():
            success_criteria = gr.Textbox(show_label=False, placeholder="What are your success criteria?")
    with gr.Row():
        go_button = gr.Button("Go!", variant="primary")
    ui.load(setup_ui, [], [sk_state])
    go_button.click(process, [sk_state, message, success_criteria, chatbot], [chatbot, sk_state])

ui.launch(theme=gr.themes.Default(primary_hue="emerald"))

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations</h2>
            <span style="color:#00cc00;">You have built the Sidekick, and with it you have travelled the whole stack: the building blocks of Layer 1, the orchestration of LangGraph, the create_agent of Layer 3, the harness of Deep Agents, and now a real project that mixes them all. That is a serious amount of capability, and you understand every layer of it. Wonderful work this week.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make the Sidekick your own. Add the human-in-the-loop middleware to the worker in <code>sidekick.py</code> so it asks your approval before sending a push notification or writing a file. Give it a new tool of your own. Then set it a real task you care about, with a clear success criterion, and watch it work.
            </span>
        </td>
    </tr>
</table>